# 🧠 EX54: Resuming Interrupted Training

Training can be interrupted by OOM, power cuts, or cloud preemption.
`last.pt` stores the **complete training state** for seamless resumption.

## `last.pt` internals
```python
{
  'epoch':        42,     # last completed (0-based)
  'best_fitness': 0.712,  # best val score so far
  'model':        ...,    # weight state_dict
  'ema':          ...,    # EMA state_dict (→ best.pt)
  'optimizer':    ...,    # AdamW m_t, v_t tensors
  'train_args':   {...},  # original config
}
```

### Why optimizer state matters
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t \quad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$
Losing $m_t, v_t$ resets adaptive LR → temporary performance dip.

- ✅ Always resume from `last.pt`
- ❌ Never resume from `best.pt` (missing optimizer state)

## 🔗 Links
- [[EX53_Training_Settings_TH]] | [[YOLO_Learning_Plan]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import gc, os
import torch, pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from solution import resume_yolo_training
%matplotlib inline

device = "0" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Device: {device}")
print("\n--- AUDIT & INSPECTION START ---")

# Step 1: Create a checkpoint via a 2-epoch base run
print("[Step 1] Training 2-epoch baseline to create last.pt...")
base = YOLO("yolo11n.pt")
base_res = base.train(data="coco8.yaml", epochs=2, imgsz=320, batch=8, device=device, verbose=False)
ckpt = str(base_res.save_dir / "weights" / "last.pt")
print(f"Checkpoint: {ckpt}")

# Step 2: Inspect checkpoint keys
print("\n[Step 2] Inspecting checkpoint...")
ck = torch.load(ckpt, map_location="cpu")
print(f"  Keys: {list(ck.keys())}")
print(f"  Epoch completed:  {ck.get('epoch','N/A')}")
print(f"  Best fitness:     {ck.get('best_fitness',0.0):.4f}")
print(f"  Optimizer state:  {'present ✅' if ck.get('optimizer') is not None else 'MISSING ❌'}")
ta = ck.get("train_args", {})
print(f"  Config: epochs={ta.get('epochs','?')}, batch={ta.get('batch','?')}, imgsz={ta.get('imgsz','?')}")

# Step 3: Resume
print("\n[Step 3] Resuming from checkpoint...")
res = resume_yolo_training(ckpt)
print(f"Resumed run saved to: {res.save_dir}")
print("--- AUDIT & INSPECTION END ---")

csv = res.save_dir / "results.csv"
if csv.exists():
    df = pd.read_csv(csv); df.columns=[c.strip() for c in df.columns]
    loss_cols = [c for c in df.columns if "loss" in c.lower()]
    plt.figure(figsize=(8,4))
    for col in loss_cols: plt.plot(df["epoch"], df[col], label=col, linewidth=2)
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss After Resume")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

del base
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
